In [ ]:
import pandas as pd
import numpy as np
import os
import re
import string
import nltk
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from transformers import BertTokenizer, BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup, BertModel, DataCollatorWithPadding
from torch.utils.data import DataLoader, Dataset, random_split
import torch
from tqdm import tqdm
import logging
import torch.nn as nn
import torch.optim as optim
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
from transformers import LongformerForSequenceClassification
from transformers import LongformerTokenizer
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from transformers import LongformerModel, LongformerConfig

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256, use_global_attention=True, global_attention_target='cls', dynamic_padding=False):
        """
        A custom dataset for handling short texts with optional global attention.

        Args:
        - texts: List of input texts.
        - labels: List of corresponding labels.
        - tokenizer: The tokenizer to convert texts to token ids.
        - max_length: Maximum length for token sequences (used when dynamic_padding=False).
        - global_attention_target: Which tokens to assign global attention to. Options: 'cls', 'question_mark', 'custom'.
        - dynamic_padding: Whether to use dynamic padding based on the longest sequence in the batch.
        - use_global_attention: Boolean flag to enable or disable the global attention mask.
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.global_attention_target = global_attention_target
        self.dynamic_padding = dynamic_padding
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        # Dynamic or fixed padding
        padding_strategy = "longest" if self.dynamic_padding else "max_length"

        if not isinstance(text, str):
            text = str(text)  # Convert to string if not already

        # Tokenize the text with the appropriate padding and truncation
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=padding_strategy, max_length=self.max_length)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)

        # Initialize the output dictionary
        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)  # Convert labels to float for BCEWithLogitsLoss
        }

        # Add global attention mask if use_global_attention is True
        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)

            # Apply global attention based on the chosen strategy
            if self.global_attention_target == 'cls':
                # Apply global attention to the CLS token (first token)
                global_attention_mask[0] = 1
            elif self.global_attention_target == 'question_mark':
                # Apply global attention to any question marks in the text
                question_mark_token_id = self.tokenizer.convert_tokens_to_ids("?")
                question_mark_position = (input_ids == question_mark_token_id).nonzero(as_tuple=True)
                if question_mark_position[0].numel() > 0:  # If there's a question mark
                    global_attention_mask[question_mark_position[0]] = 1
            elif self.global_attention_target == False:
                # Disable global attention
                global_attention_mask = torch.zeros_like(input_ids)
            elif self.global_attention_target == 'custom':
                # Implement custom logic to apply global attention to specific tokens
                pass  # Add custom logic here

            # Add global_attention_mask to the output dictionary
            output['global_attention_mask'] = global_attention_mask

        return output


In [ ]:
# Train function with BCEWithLogitsLoss
def train_one_epoch(model, data_loader, criterion, optimizer, device, scheduler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    # Initialize the progress bar
    pbar = tqdm(enumerate(data_loader), total=len(data_loader), desc="Training")

    for batch_idx, data in pbar:
        input_ids = data['input_ids'].to(device)
        attention_mask = data['attention_mask'].to(device)
        labels = data['labels'].float().to(device)  # Convert labels to float for BCEWithLogitsLoss

        # Check if global_attention_mask is present in the batch
        global_attention_mask = data.get('global_attention_mask', None)
        if global_attention_mask is not None:
            global_attention_mask = global_attention_mask.to(device)

        optimizer.zero_grad()

        # Forward pass
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)

        # Compute the loss (BCEWithLogitsLoss expects logits and float labels)
        loss = criterion(outputs.logits.view(-1), labels.float().view(-1))
        loss.backward()
        optimizer.step()
        scheduler.step()

        running_loss += loss.item() * input_ids.size(0)

        # Sigmoid to convert logits to probabilities
        probabilities = torch.sigmoid(outputs.logits.squeeze())

        # Predictions (>= 0.5 is classified as class 1)
        predicted = (probabilities >= 0.5).float()

        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update the progress bar
        pbar.set_postfix({
            'loss': running_loss / ((batch_idx + 1) * data_loader.batch_size),
            'accuracy': 100 * correct / total
        })

    epoch_loss = running_loss / len(data_loader.dataset)
    epoch_accuracy = 100 * correct / total
    return epoch_loss, epoch_accuracy


In [ ]:
def evaluate(model, data_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    # Initialize the progress bar
    pbar = tqdm(enumerate(data_loader), total=len(data_loader), desc="Evaluating")

    with torch.no_grad():
        for batch_idx, data in pbar:
            input_ids = data['input_ids'].to(device)
            attention_mask = data['attention_mask'].to(device)
            labels = data['labels'].float().to(device)  # Ensure labels are in float

            # Check if global_attention_mask is present in the batch
            global_attention_mask = data.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)

            # Use view(-1) instead of squeeze() to avoid removing batch dimension when it's 1
            loss = criterion(outputs.logits.view(-1), labels.view(-1))  # Ensure matching shapes for loss calculation
            running_loss += loss.item() * input_ids.size(0)

            # Sigmoid to convert logits to probabilities
            probabilities = torch.sigmoid(outputs.logits.view(-1))

            # Predictions (>= 0.5 is classified as class 1)
            predicted = (probabilities >= 0.5).float()

            correct += (predicted == labels.view(-1)).sum().item()
            total += labels.size(0)

            # Update the progress bar
            pbar.set_postfix({
                'loss': running_loss / ((batch_idx + 1) * data_loader.batch_size),
                'accuracy': 100 * correct / total
            })

    epoch_loss = running_loss / len(data_loader.dataset)
    epoch_accuracy = 100 * correct / total
    return epoch_loss, epoch_accuracy


In [ ]:
from torch.nn.utils.rnn import pad_sequence

def custom_collate_fn(batch):
    """
    Custom collate function to dynamically pad sequences within a batch to match the longest sequence.
    """
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    global_attention_mask = [item['global_attention_mask'] for item in batch]
    labels = torch.tensor([item['labels'] for item in batch], dtype=torch.float)  # Convert labels to float for BCEWithLogitsLoss

    # Pad the sequences in the batch to match the longest sequence in the batch
    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    attention_mask = pad_sequence(attention_mask, batch_first=True, padding_value=0)
    global_attention_mask = pad_sequence(global_attention_mask, batch_first=True, padding_value=0)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'global_attention_mask': global_attention_mask,
        'labels': labels  # Labels now in float format
    }


In [ ]:
def get_predictions(model, data_loader, device):
    """Gets predictions from the model on a given data loader."""
    model.eval()
    predictions = []
    true_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Check if global_attention_mask is present in the batch
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)

            # Forward pass
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)

            # Apply sigmoid to get probabilities
            probabilities = torch.sigmoid(outputs.logits)

            # Convert probabilities to binary predictions (threshold = 0.5)
            predicted = (probabilities >= 0.5).float()

            # Ensure predictions and labels are 1-dimensional arrays before extending
            predictions.extend(predicted.view(-1).cpu().numpy())  # Flatten to 1D
            true_labels.extend(labels.view(-1).cpu().numpy())     # Flatten to 1D

    return np.array(predictions), np.array(true_labels)


In [ ]:
# Initialize the Longformer model for classification
model = LongformerForSequenceClassification.from_pretrained('allenai/longformer-base-4096', attention_window = 128, num_labels=1)
tokenizer = LongformerTokenizer.from_pretrained('allenai/longformer-base-4096')

# Move the model to the appropriate device (GPU or CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/597M [00:00<?, ?B/s]

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

LongformerForSequenceClassification(
  (longformer): LongformerModel(
    (embeddings): LongformerEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(4098, 768, padding_idx=1)
    )
    (encoder): LongformerEncoder(
      (layer): ModuleList(
        (0-11): 12 x LongformerLayer(
          (attention): LongformerAttention(
            (self): LongformerSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (query_global): Linear(in_features=768, out_features=768, bias=True)
              (key_global): Linear(in_features=768, out_features=768, bias=True)
          

In [ ]:
# 0.628
mimic_train = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_train.csv')
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/new_berkeley_test.csv')

In [ ]:
train_texts = mimic_train['text'].to_numpy()
train_labels = mimic_train['labels'].to_numpy()
val_texts = mimic_test['text'].to_numpy()
val_labels = mimic_test['labels'].to_numpy()

In [ ]:
# Create Dataset classes for training and validation sets
train_dataset = TextDataset(train_texts, train_labels, tokenizer, max_length=128, use_global_attention=False, global_attention_target=False)
val_dataset = TextDataset(val_texts, val_labels, tokenizer, max_length=128, use_global_attention=False, global_attention_target=False)

# Create DataLoader for training and validation sets
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

In [ ]:
# Define the optimizer and loss function
num_epochs = 8
#model.config.attention_probs_dropout_prob = 0.2  # Increasing attention dropout to 0.2
#model.config.hidden_dropout_prob = 0.2  # Increasing hidden dropout to 0.2
optimizer = optim.AdamW(model.parameters(), lr=1e-5)
criterion = nn.BCEWithLogitsLoss()
total_steps = len(train_loader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

In [ ]:
save_directory_model = '/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain'
os.makedirs(save_directory_model, exist_ok=True)

In [ ]:
best_val_accuracy = 0.0

# simple pre-process
for epoch in range(num_epochs):
    print(f'Epoch [{epoch + 1}/{num_epochs}]')
    train_loss, train_accuracy = train_one_epoch(model, train_loader, criterion, optimizer, device, scheduler)
    print(f'Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%')

    val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%')

    # Save the model only if the validation accuracy has improved
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        model.save_pretrained(save_directory_model)
        tokenizer.save_pretrained(save_directory_model)
        print(f"Model saved at epoch {epoch + 1} with improved validation accuracy: {val_accuracy:.2f}%")

        # Get predictions on the combined validation set
        predictions, true_labels = get_predictions(model, val_loader, device)

        # Calculate confusion matrix
        cm = confusion_matrix(true_labels, predictions)
        print("Confusion Matrix:")
        print(cm)

        # Calculate precision, recall, F1-score
        report = classification_report(true_labels, predictions)
        print("Classification Report:")
        print(report)

Epoch [1/8]


Training: 100%|██████████| 87/87 [02:49<00:00,  1.94s/it, loss=0.482, accuracy=74.5]


Training Loss: 0.4820, Training Accuracy: 74.55%


Evaluating: 100%|██████████| 22/22 [00:14<00:00,  1.51it/s, loss=0.32, accuracy=85.1]


Validation Loss: 0.3237, Validation Accuracy: 85.10%
Model saved at epoch 1 with improved validation accuracy: 85.10%
Confusion Matrix:
[[3049  474]
 [ 355 1687]]
Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.87      0.88      3523
         1.0       0.78      0.83      0.80      2042

    accuracy                           0.85      5565
   macro avg       0.84      0.85      0.84      5565
weighted avg       0.85      0.85      0.85      5565

Epoch [2/8]


...

Validation Loss: 0.3053, Validation Accuracy: 87.80%
Model saved at epoch 7 with improved validation accuracy: 87.80%
Confusion Matrix:
[[3210  313]
 [ 366 1676]]
Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.91      0.90      3523
         1.0       0.84      0.82      0.83      2042

    accuracy                           0.88      5565
   macro avg       0.87      0.87      0.87      5565
weighted avg       0.88      0.88      0.88      5565

Epoch [8/8]


Training: 100%|██████████| 87/87 [02:44<00:00,  1.89s/it, loss=0.185, accuracy=92.8]


Training Loss: 0.1850, Training Accuracy: 92.83%


Evaluating: 100%|██████████| 22/22 [00:14<00:00,  1.54it/s, loss=0.311, accuracy=87.5]

Validation Loss: 0.3145, Validation Accuracy: 87.49%
